# 3-3 Googleトレンドで世の中の関心を測る / Measuring Public Interest with Google Trends

『AIに頼んで動かす Python実務データ分析』第3章3節の参照用ノートブックです。
Reference notebook for Chapter 3, Section 3 of *Data Analysis with AI and Python*.

**使い方 / How to use**
1. 本文の手順で、Googleトレンドから2つのCSVをダウンロードします。 / Download two CSV files from Google Trends as described in the book.
   - `trends_12m.csv`：ユニクロ, UNIQLO, ジーユー, GU（日本、過去12か月） / Japan, past 12 months
   - `trends_5y.csv`：同じ4語（日本、過去5年間） / the same four terms, Japan, past 5 years
2. 左のファイルパネルから2つのファイルをアップロードします。 / Upload both files from the Files panel.
3. `LANG` を選んで、すべてのセルを上から実行します。 / Choose `LANG` and run all cells.

データの出典 / Data source: Google Trends (https://trends.google.com/)

In [ ]:
# ===== 設定 / Settings =====
LANG = "ja"   # "ja" = 日本語 / "en" = English
SAVE_FIGURES = True
BASE_URL = "https://raw.githubusercontent.com/rekishi-data/ai-python-data-analysis/main/data/"
FILE_12M = "trends_12m.csv"
FILE_5Y = "trends_5y.csv"
BRANDS = ["ユニクロ", "GU"]   # 3-3-4以降で比べる2つの検索語 / the two terms compared from 3-3-4

In [ ]:
import os, re, subprocess, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

JP_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if not os.path.exists(JP_FONT):   # 英語版でも日本語の検索語を表示できるように / needed for Japanese search terms in both editions
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True, check=True)
fm.fontManager.addfont(JP_FONT)
plt.rcParams["font.family"] = [fm.FontProperties(fname=JP_FONT).get_name()]
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "black", "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
FIG_W = 4.5

def save(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(f"figures/{LANG}", exist_ok=True)
        fig.savefig(f"figures/{LANG}/{name}.png", bbox_inches="tight")
    plt.show()

L = {"ja": dict(interest="検索インタレスト", avg="平均の検索インタレスト", month="月",
                index="季節指数（年平均＝100）", months=[f"{m}月" for m in range(1, 13)], ma_w="13週移動平均", ma_m="3か月移動平均"),
     "en": dict(interest="Search interest", avg="Average search interest", month="Month",
                index="Seasonal index (annual mean = 100)", months=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], ma_w="13-week moving average", ma_m="3-month moving average")}[LANG]
# 英語版の表示名 / Display names for the English edition
NAMES_EN = {"ユニクロ": "Uniqlo (katakana)", "UNIQLO": "UNIQLO (Latin)", "ジーユー": "GU (katakana)",
            "GU": "GU (Latin)", "無印良品": "Muji", "ZARA": "ZARA", "H&M": "H&M"}
name = lambda k: NAMES_EN.get(k, k) if LANG == "en" else k

## GoogleトレンドのCSVを読み込む / Loading a Google Trends CSV

Googleトレンドから書き出したCSVは、先頭に説明の行があり、「1未満」を表す `<1` という文字も入っています。そのまま `pd.read_csv` で読むとうまくいかないので、次の関数で整えます。
Google Trends CSV files start with description lines and contain `<1` for values below 1, so we clean them with this function.

In [ ]:
import urllib.request
# 自分で取得したCSVがなければ、本書で使ったデータ（2026年9月24日取得）を読み込む
# If you have not uploaded your own CSV files, download the data used in the book (retrieved on 24 Sep 2026)
for f in [FILE_12M, FILE_5Y]:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
def load_trends_csv(path):
    """GoogleトレンドのCSVを読み込み、日付を行・検索語を列にした表を返す
    Read a Google Trends CSV and return a table with dates as rows and search terms as columns."""
    lines = open(path, encoding="utf-8-sig").read().splitlines()
    # 次の行が日付で始まる行を、見出しの行とみなす / The header is the line just before the first date
    header = next(i for i in range(len(lines) - 1)
                  if "," in lines[i] and re.match(r"^\"?\d{4}-\d{2}", lines[i + 1]))
    df = pd.read_csv(path, skiprows=header, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"].astype(str).str[:10])
    df = df.set_index("date")
    # 「ユニクロ: (日本)」→「ユニクロ」 / "UNIQLO: (Japan)" -> "UNIQLO"
    df.columns = [re.sub(r":\s*\(.*\)\s*$", "", c).strip() for c in df.columns]
    # 「<1」は1未満なので0として扱う / Treat "<1" as 0
    return df.replace("<1", 0).apply(pd.to_numeric)

kw = load_trends_csv(FILE_12M)
y5 = load_trends_csv(FILE_5Y).iloc[:-1]   # 最後の月は集計途中なので除く / drop the last (partial) month
print(kw.shape, kw.index.min().date(), "–", kw.index.max().date())
print(y5.shape, y5.index.min().date(), "–", y5.index.max().date())
y5.head()

## 3-3-3 キーワード選びが結果を決める / Your choice of keywords decides the result

In [ ]:
styles = ["-", "--", ":", "-."]
fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
ends = {}
for (col, s) in zip(kw.columns, styles):
    ax.plot(kw.index, kw[col], color="black", ls=s, lw=1.1)
    ends[col] = kw[col].iloc[-4:].mean()
# 右端のラベルが重ならないよう、8以上の間隔をあける / keep end labels at least 8 apart
pos, last = {}, -99
for col, y in sorted(ends.items(), key=lambda t: t[1]):
    last = max(y, last + 8); pos[col] = last
for col, y in pos.items():
    ax.text(kw.index[-1] + pd.Timedelta(days=5), y, name(col), fontsize=7, va="center")
ax.set_ylabel(L["interest"]); ax.set_ylim(0, 105)
ax.set_xlim(kw.index[0], kw.index[-1] + pd.Timedelta(days=80))
fig.autofmt_xdate(rotation=45)
save(fig, "fig3-3-2_keywords")
kw.mean().round(1).rename(name)

### 5年間で見る / Looking at five years (Figure 3-3-3)

In [ ]:
cols = list(y5.columns)
fig, axs = plt.subplots(len(cols), 1, figsize=(FIG_W, 3.6), sharex=True, sharey=True)
for ax, c in zip(axs, cols):
    ax.plot(y5.index, y5[c], color="#999999", lw=0.7, marker="o", ms=1.5)
    ax.plot(y5.index, y5[c].rolling(3, center=True).mean(), color="black", lw=1.2)
    ax.text(0.01, 0.78, name(c), transform=ax.transAxes, fontsize=7.5, fontweight="bold")
    ax.set_ylim(0, 105); ax.set_yticks([0, 50, 100]); ax.tick_params(labelsize=6.5)
axs[-1].text(0.99, 0.72, f"— {L['ma_m']}", transform=axs[-1].transAxes, fontsize=6.5, ha="right")
fig.supylabel(L["interest"], fontsize=7.5)
save(fig, "fig3-3-3_five_years")

## 3-3-4 ユニクロとGUの5年間 / Uniqlo and GU over five years

In [ ]:
first12, last12 = y5.iloc[:12], y5.iloc[-12:]
comp = pd.DataFrame({"period_first": f"{first12.index[0]:%Y-%m}–{first12.index[-1]:%Y-%m}",
                     "first_12m": first12[BRANDS].mean().round(1),
                     "period_last": f"{last12.index[0]:%Y-%m}–{last12.index[-1]:%Y-%m}",
                     "last_12m": last12[BRANDS].mean().round(1)})
comp["change_%"] = ((comp.last_12m / comp.first_12m - 1) * 100).round(0)
comp.rename(index=name)

## 3-3-5 季節のパターンを読む / Reading seasonal patterns (Figure 3-3-4)

In [ ]:
# 12か月ごとの区切り（期間の最初の月から）で平均を100とした指数にし、月ごとに平均する
# Index each 12-month block (from the first month) to its mean (=100), then average by calendar month
data = y5[BRANDS]
block = np.arange(len(data)) // 12
idx = data.groupby(block).transform(lambda x: x / x.mean() * 100)
season = idx.groupby(idx.index.month).mean().T.round(0)
season.columns = L["months"]

fig, ax = plt.subplots(figsize=(FIG_W, 1.2))
im = ax.imshow(season.values, cmap="Greys", vmin=70, vmax=150, aspect="auto")
nr, nc = im.get_array().shape
ax.set_xticks(np.arange(-0.5, nc, 1), minor=True); ax.set_yticks(np.arange(-0.5, nr, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.8); ax.tick_params(which="minor", length=0)
for i in range(season.shape[0]):
    for j in range(12):
        v = season.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6.5, color="white" if v > 120 else "black")
ax.set_xticks(range(12), season.columns, fontsize=6.5); ax.set_yticks(range(len(season)), [name(c) for c in season.index])
for sp in ax.spines.values(): sp.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.9).ax.tick_params(labelsize=6.5)
save(fig, "fig3-3-4_seasonality")
season.rename(index=name)